# Gas benchmark data - EDA v3

#### Maria Silva, January 2026

In [1]:
import os
import sys
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Load data

This data was generated from running the `estimate_opcode_run_times` script in this repo. The raw data comes from the repricing benchmarks between done between 2026-01-10 and 2026-01-22.

In [2]:
# Main directories
current_path = os.getcwd()
repo_dir = os.path.abspath(os.path.join(current_path, ".."))
report_dir = os.path.join(
    repo_dir, "reports", "opcode_run_times_estimation", "2026-01-10_2026-01-22"
)
src_dir = os.path.join(repo_dir, "src")

In [3]:
sys.path.append(src_dir)
import operation_types

In [4]:
df = pd.read_csv(os.path.join(report_dir, "gas_bench_data.csv"))

## Opcode counts

In [5]:
df["opcount"].value_counts()

opcount
14000    9982
70000    8547
21000    8484
35000    8309
7000     8304
         ... 
280       105
1125       36
625        36
375        36
875        36
Name: count, Length: 178, dtype: int64

In [6]:
0 in df["opcount"].value_counts().index

True

Ok, there seems to have some issues with the opcode counts. We shouldn't have zero opcodes.

In [7]:
df[df["opcount"]==0]["test_name"].unique()

array(['test_bls12_g1_msm', 'test_bls12_g2_msm', 'test_bls12_pairing',
       'test_alt_bn128_benchmark'], dtype=object)

They seem to be all related to the new bls12 and alt_bn128 tests.

In [8]:
df[df["opcount"]==0]

,client_name,run_duration_ms,opcount,test_file,test_name,test_opcode,test_params
23,geth,1082.57820,0,test_bls12_381,test_bls12_g1_msm,BLS12_G1MSM,k_128
55,geth,56.79869,0,test_bls12_381,test_bls12_g2_msm,BLS12_G2MSM,k_128
122,geth,627.49980,0,test_bls12_381,test_bls12_g2_msm,BLS12_G2MSM,k_64
179,geth,1164.45360,0,test_bls12_381,test_bls12_g2_msm,BLS12_G2MSM,k_16
243,geth,255.70581,0,test_bls12_381,test_bls12_pairing,BLS12_PAIRING_CHECK,num_pairs_24
...,...,...,...,...,...,...,...
403158,geth,923.68570,0,test_alt_bn128,test_alt_bn128_benchmark,ECPAIRING,num_pairs_3
403164,geth,1640.93310,0,test_bls12_381,test_bls12_g1_msm,BLS12_G1MSM,k_64
403169,geth,602.38600,0,test_alt_bn128,test_alt_bn128_benchmark,ECPAIRING,num_pairs_24
403207,geth,128.92188,0,test_bls12_381,test_bls12_g2_msm,BLS12_G2MSM,k_128


What about empty opcode counts?

In [9]:
df["opcount"].isna().sum()/len(df)

np.float64(0.0)

Ok, no empty opcode counts now.

## Missing opcodes

In [10]:
benchmarked_operations = set(df["test_opcode"].unique().tolist())
all_operations = set(operation_types.ALL_OPERATIONS)

In [11]:
misnamed_operations = benchmarked_operations - all_operations
print(f"Number of misnamed operations: {len(misnamed_operations)}")
misnamed_operations

Number of misnamed operations: 0


set()

In [12]:
missing_operations = all_operations - benchmarked_operations
print(f"Number of missing operations: {len(missing_operations)-len(misnamed_operations)}")
missing_operations

Number of missing operations: 11


{'BLOBHASH',
 'INVALID',
 'JUMP',
 'MODEXP',
 'P256VERIFY',
 'POP',
 'PREVRANDAO',
 'RETURN',
 'REVERT',
 'SELFDESTRUCT',
 'STOP'}

## Test configurations

In [13]:
test_configs = df[df["test_params"].notna()].drop_duplicates("test_params")[
    ["test_opcode", "test_params"]
].sort_values("test_opcode")

test_configs

,test_opcode,test_params
111,BLAKE2F,num_rounds_1
71,BLAKE2F,num_rounds_24
941,BLAKE2F,num_rounds_6
1621,BLAKE2F,num_rounds_12
227102,BLAKE2F,blake2f
...,...,...
2694,SSTORE,SSTORE_new
171,SSTORE,SSTORE new value
1034,SSTORE,SSTORE same value
75,TLOAD,fixed_value_True-fixed_key_True


In [14]:
non_simple_operations = set(operation_types.ALL_OPERATIONS).difference(operation_types.SIMPLE_COMPUTE)
configed_operations = set(test_configs["test_opcode"].unique())
configed_operations

print(f"Operations without configs: {len(non_simple_operations)-len(misnamed_operations)}")
non_simple_operations.difference(configed_operations).difference(missing_operations)

Operations without configs: 47


{'BALANCE',
 'CALL',
 'CALLCODE',
 'DELEGATECALL',
 'EXP',
 'EXTCODESIZE',
 'SHA2-256',
 'STATICCALL'}

## Estimation issues with simple opcodes

In [15]:
results_df = pd.read_csv(os.path.join(report_dir, "simple_opcodes_results.csv"))
results_df

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high
0,ADD,geth,31.215030,3.670085e-14,0.984615,0.984560,0.024309,6.038810e-255,0.023951,0.024667
1,ADD,besu,43.126270,7.682673e-01,0.677721,0.676566,0.164852,1.448115e-70,0.151454,0.178249
2,ADD,reth,22.196835,2.654642e-03,0.970155,0.969979,0.025280,8.324366e-131,0.024607,0.025953
3,ADD,nethermind,28.672421,2.675739e-17,0.981901,0.981836,0.018159,4.230335e-245,0.017868,0.018449
4,ADD,erigon,31.966839,2.547706e-15,0.999631,0.999618,0.026043,1.283229e-49,0.025849,0.026236
...,...,...,...,...,...,...,...,...,...,...
500,XOR,geth,30.717899,2.256452e-07,0.966515,0.966399,0.024312,1.811482e-214,0.023788,0.024837
501,XOR,besu,34.096064,4.086656e-07,0.995142,0.995125,0.073556,0.000000e+00,0.072959,0.074152
502,XOR,reth,31.467019,5.198916e-36,0.980452,0.980342,0.008609,4.891289e-154,0.008429,0.008788
503,XOR,nethermind,22.053947,3.470890e-01,0.563437,0.561922,0.020789,9.161892e-54,0.018666,0.022911


In [16]:
all_opcodes = set(operation_types.SIMPLE_COMPUTE)
non_missing_operations = all_opcodes - missing_operations

estimated_operations_without_errors = set(results_df["opcode"].unique().tolist())
estimated_operations_with_errors = (
    non_missing_operations - estimated_operations_without_errors
)

print(
    f"Number of operations with estimation errors: {len(estimated_operations_with_errors)}"
)
estimated_operations_with_errors

Number of operations with estimation errors: 0


set()

In [17]:
estimation_by_client = results_df.groupby("opcode")["client"].nunique()
max_clients = estimation_by_client.max()
print(f"There is a max of {max_clients} clients with estimations.")
opcodes_with_missing_clients = estimation_by_client[estimation_by_client<5].index
print(
    f"Number of operations with estimation errors: {len(opcodes_with_missing_clients)}"
)
opcodes_with_missing_clients

There is a max of 5 clients with estimations.
Number of operations with estimation errors: 70


Index(['DUP1', 'DUP10', 'DUP11', 'DUP12', 'DUP13', 'DUP14', 'DUP15', 'DUP16',
       'DUP2', 'DUP3', 'DUP4', 'DUP5', 'DUP6', 'DUP7', 'DUP8', 'DUP9',
       'GASPRICE', 'MSIZE', 'ORIGIN', 'PUSH0', 'PUSH1', 'PUSH10', 'PUSH11',
       'PUSH12', 'PUSH13', 'PUSH14', 'PUSH15', 'PUSH16', 'PUSH17', 'PUSH18',
       'PUSH19', 'PUSH2', 'PUSH20', 'PUSH21', 'PUSH22', 'PUSH23', 'PUSH24',
       'PUSH25', 'PUSH26', 'PUSH27', 'PUSH28', 'PUSH29', 'PUSH3', 'PUSH30',
       'PUSH31', 'PUSH32', 'PUSH4', 'PUSH5', 'PUSH6', 'PUSH7', 'PUSH8',
       'PUSH9', 'SWAP1', 'SWAP10', 'SWAP11', 'SWAP12', 'SWAP13', 'SWAP14',
       'SWAP15', 'SWAP16', 'SWAP2', 'SWAP3', 'SWAP4', 'SWAP5', 'SWAP6',
       'SWAP7', 'SWAP8', 'SWAP9', 'TLOAD', 'TSTORE'],
      dtype='object', name='opcode')

#### Negative slopes

In [18]:
results_df[results_df["slope"]<0]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high


#### Non-significant slopes (by p-value)

In [19]:
results_df[results_df["slope_pvalue"]>0.05]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high


In [20]:
non_significant_ops = results_df[results_df["slope_pvalue"]>0.05]["opcode"].unique().tolist()
non_significant_ops

[]

#### Small r-squared values (besides non-significant slopes)

In [21]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high
31,BLOCKHASH,besu,28.983021,0.030439,0.429451,0.429057,0.071180,1.150467e-178,0.066951,0.075409
40,CALLDATALOAD,geth,30.955052,0.254822,0.498211,0.497777,0.029696,1.326674e-175,0.027978,0.031414


In [22]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]["opcode"].unique()

array(['BLOCKHASH', 'CALLDATALOAD'], dtype=object)

## Estimation issues with memory opcodes

In [23]:
results_df = pd.read_csv(os.path.join(report_dir, "memory_opcodes_results.csv"))
results_df

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,...,calldata_size_conf_int_low,calldata_size_conf_int_high,code_size,code_size_pvalue,code_size_conf_int_low,code_size_conf_int_high,return_size,return_size_pvalue,return_size_conf_int_low,return_size_conf_int_high
0,KECCAK256,geth,-68.402653,2.108773e-04,0.705613,0.705423,0.564278,0.000000e+00,0.553551,0.575005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KECCAK256,besu,-50.007426,3.986252e-02,0.752880,0.752720,0.855596,0.000000e+00,0.841449,0.869744,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KECCAK256,reth,139.725011,2.250594e-08,0.514548,0.514041,0.055097,1.196968e-13,0.040602,0.069591,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,KECCAK256,nethermind,134.125738,3.552166e-10,0.516526,0.516213,0.064541,3.682979e-24,0.052132,0.076949,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,KECCAK256,erigon,-55.470305,2.981277e-01,0.714546,0.712747,0.531128,8.413975e-128,0.500085,0.562171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,MCOPY,geth,40.257728,3.157968e-170,0.986070,0.986066,0.075546,0.000000e+00,0.075277,0.075815,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MCOPY,besu,115.399906,6.767016e-178,0.947883,0.947868,0.109568,0.000000e+00,0.108815,0.110322,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,MCOPY,reth,47.244531,0.000000e+00,0.931166,0.931134,0.018079,0.000000e+00,0.017889,0.018269,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MCOPY,nethermind,92.520829,1.059929e-208,0.879109,0.879074,0.048772,0.000000e+00,0.048218,0.049327,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,MLOAD,geth,31.241856,1.685464e-46,0.982614,0.982590,0.037470,0.000000e+00,0.037213,0.037727,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
all_opcodes = set(operation_types.MEMORY_COMPUTE)
non_missing_operations = all_opcodes - missing_operations

estimated_operations_without_errors = set(results_df["opcode"].unique().tolist())
estimated_operations_with_errors = (
    non_missing_operations - estimated_operations_without_errors
)

print(
    f"Number of operations with estimation errors: {len(estimated_operations_with_errors)}"
)
estimated_operations_with_errors

Number of operations with estimation errors: 0


set()

In [25]:
estimation_by_client = results_df.groupby("opcode")["client"].nunique()
max_clients = estimation_by_client.max()
print(f"There is a max of {max_clients} clients with estimations.")
opcodes_with_missing_clients = estimation_by_client[estimation_by_client<5].index
print(
    f"Number of operations with estimation errors: {len(opcodes_with_missing_clients)}"
)
opcodes_with_missing_clients

There is a max of 5 clients with estimations.
Number of operations with estimation errors: 4


Index(['MCOPY', 'MLOAD', 'MSTORE', 'MSTORE8'], dtype='object', name='opcode')

#### Negative features

In [26]:
pvalue_cols = [i for i in results_df.columns if ("_pvalue" in i) & ("intercept" not in i)]
coef_cols = [i.replace("_pvalue", "") for i in pvalue_cols]
coef_cols

['slope',
 'mem_size',
 'msg_size',
 'copy_size',
 'calldata_size',
 'code_size',
 'return_size']

In [27]:
features_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=coef_cols,
    var_name="feature",
    value_name="value"
).dropna()
features_df[features_df["value"]<0].sort_values("opcode")

,opcode,client,feature,value
61,CALLDATACOPY,erigon,mem_size,-0.000002
165,CALLDATACOPY,geth,calldata_size,-0.003455
166,CALLDATACOPY,besu,calldata_size,-0.082717
167,CALLDATACOPY,reth,calldata_size,-0.000536
168,CALLDATACOPY,nethermind,calldata_size,-0.028926
169,CALLDATACOPY,erigon,calldata_size,-0.004156
65,CODECOPY,nethermind,mem_size,-0.001038
66,CODECOPY,erigon,mem_size,-0.000848
207,CODECOPY,besu,code_size,-0.008485
209,CODECOPY,nethermind,code_size,-0.001187


#### Non-significant features (by p-value)

In [28]:
results_df[results_df["slope_pvalue"]>0.05]

pvalue_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=pvalue_cols,
    var_name="feature",
    value_name="pvalue"
).dropna()
pvalue_df[pvalue_df["pvalue"]>0.05].sort_values("opcode")

,opcode,client,feature,pvalue
167,CALLDATACOPY,reth,calldata_size_pvalue,0.826206
165,CALLDATACOPY,geth,calldata_size_pvalue,0.247206
169,CALLDATACOPY,erigon,calldata_size_pvalue,0.381977
61,CALLDATACOPY,erigon,mem_size_pvalue,0.677185
65,CODECOPY,nethermind,mem_size_pvalue,0.762226
66,CODECOPY,erigon,mem_size_pvalue,0.823027
64,CODECOPY,reth,mem_size_pvalue,0.622887
208,CODECOPY,reth,code_size_pvalue,0.091726
62,CODECOPY,geth,mem_size_pvalue,0.214329
63,CODECOPY,besu,mem_size_pvalue,0.763837


#### Small r-squared values (besides non-significant slopes)

In [29]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,...,calldata_size_conf_int_low,calldata_size_conf_int_high,code_size,code_size_pvalue,code_size_conf_int_low,code_size_conf_int_high,return_size,return_size_pvalue,return_size_conf_int_low,return_size_conf_int_high


In [30]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]["opcode"].unique()

array([], dtype=object)

## Estimation issues with log opcodes

In [31]:
results_df = pd.read_csv(os.path.join(report_dir, "log_opcodes_results.csv"))
results_df

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,log_size,log_size_pvalue,log_size_conf_int_low,log_size_conf_int_high,mem_size,mem_size_pvalue,mem_size_conf_int_low,mem_size_conf_int_high
0,LOG0,geth,87.282846,1.718058e-126,0.865073,0.864986,1.474600,0.000000e+00,1.455230,1.493970,0.117833,3.716215e-136,0.108839,0.126826,0.002059,5.657240e-01,-0.004969,0.009087
1,LOG0,besu,74.411989,2.577295e-98,0.898134,0.898068,1.680516,0.000000e+00,1.661608,1.699423,0.128218,3.503358e-166,0.119439,0.136996,0.008765,1.228044e-02,0.001905,0.015625
2,LOG0,reth,49.681034,2.712372e-107,0.583468,0.583033,0.374100,0.000000e+00,0.362247,0.385952,0.077477,5.240849e-149,0.071974,0.082980,0.000123,9.554102e-01,-0.004178,0.004423
3,LOG0,nethermind,56.350661,1.672280e-137,0.824012,0.823899,0.791164,0.000000e+00,0.779211,0.803117,0.075275,4.066492e-145,0.069725,0.080825,0.011402,2.652635e-07,0.007065,0.015739
4,LOG0,erigon,37.831742,9.214542e-02,0.978279,0.977503,2.069273,7.858199e-25,1.949911,2.188635,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
5,LOG1,geth,63.818889,4.202200e-84,0.835830,0.835724,2.483772,0.000000e+00,2.448818,2.518726,0.149284,1.111943e-275,0.141588,0.156980,0.001682,5.800451e-01,-0.004276,0.007640
6,LOG1,besu,60.209932,7.554341e-72,0.887860,0.887787,3.083594,0.000000e+00,3.047735,3.119453,0.146038,1.029583e-253,0.138143,0.153933,0.005931,5.719040e-02,-0.000181,0.012043
7,LOG1,reth,39.682596,1.989566e-79,0.514314,0.513808,0.623197,0.000000e+00,0.601054,0.645340,0.087446,1.056649e-225,0.082570,0.092321,0.000567,7.684306e-01,-0.003207,0.004341
8,LOG1,nethermind,50.916762,6.518427e-133,0.824318,0.824205,1.473525,0.000000e+00,1.451708,1.495343,0.082626,3.841031e-223,0.077823,0.087430,0.011210,3.670784e-09,0.007491,0.014929
9,LOG1,erigon,-11.234263,7.305452e-01,0.942997,0.940717,3.885694,4.561856e-17,3.492177,4.279210,0.000000,NaN,0.000000,0.000000,0.000000,NaN,0.000000,0.000000


In [32]:
all_opcodes = set(operation_types.LOG)
non_missing_operations = all_opcodes - missing_operations

estimated_operations_without_errors = set(results_df["opcode"].unique().tolist())
estimated_operations_with_errors = (
    non_missing_operations - estimated_operations_without_errors
)

print(
    f"Number of operations with estimation errors: {len(estimated_operations_with_errors)}"
)
estimated_operations_with_errors

Number of operations with estimation errors: 0


set()

In [33]:
estimation_by_client = results_df.groupby("opcode")["client"].nunique()
max_clients = estimation_by_client.max()
print(f"There is a max of {max_clients} clients with estimations.")
opcodes_with_missing_clients = estimation_by_client[estimation_by_client<5].index
print(
    f"Number of operations with estimation errors: {len(opcodes_with_missing_clients)}"
)
opcodes_with_missing_clients

There is a max of 5 clients with estimations.
Number of operations with estimation errors: 3


Index(['LOG2', 'LOG3', 'LOG4'], dtype='object', name='opcode')

#### Negative features

In [34]:
pvalue_cols = [i for i in results_df.columns if ("_pvalue" in i) & ("intercept" not in i)]
coef_cols = [i.replace("_pvalue", "") for i in pvalue_cols]
coef_cols

['slope', 'log_size', 'mem_size']

In [35]:
features_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=coef_cols,
    var_name="feature",
    value_name="value"
).dropna()
features_df[features_df["value"]<0].sort_values("opcode")

,opcode,client,feature,value
56,LOG2,reth,mem_size,-0.000622
60,LOG3,reth,mem_size,-0.000607


#### Non-significant features (by p-value)

In [36]:
results_df[results_df["slope_pvalue"]>0.05]

pvalue_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=pvalue_cols,
    var_name="feature",
    value_name="pvalue"
).dropna()
pvalue_df[pvalue_df["pvalue"]>0.05].sort_values("opcode")

,opcode,client,feature,pvalue
44,LOG0,geth,mem_size_pvalue,0.565724
46,LOG0,reth,mem_size_pvalue,0.955410
49,LOG1,geth,mem_size_pvalue,0.580045
50,LOG1,besu,mem_size_pvalue,0.057190
51,LOG1,reth,mem_size_pvalue,0.768431
54,LOG2,geth,mem_size_pvalue,0.309441
55,LOG2,besu,mem_size_pvalue,0.064617
56,LOG2,reth,mem_size_pvalue,0.704107
58,LOG3,geth,mem_size_pvalue,0.861914
59,LOG3,besu,mem_size_pvalue,0.178179


#### Small r-squared values (besides non-significant slopes)

In [37]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,log_size,log_size_pvalue,log_size_conf_int_low,log_size_conf_int_high,mem_size,mem_size_pvalue,mem_size_conf_int_low,mem_size_conf_int_high


In [38]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]["opcode"].unique()

array([], dtype=object)

## Estimation issues with precompiles

In [39]:
results_df = pd.read_csv(os.path.join(report_dir, "precompiles_results.csv"))
results_df

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,...,size_conf_int_low,size_conf_int_high,num_pairs,num_pairs_pvalue,num_pairs_conf_int_low,num_pairs_conf_int_high,k,k_pvalue,k_conf_int_low,k_conf_int_high
0,ECRECOVER,geth,38.716682,2.463146e-03,0.991978,0.991950,48.183547,7.797417e-304,47.681002,48.686093,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ECRECOVER,besu,30.088828,1.058785e-01,0.983642,0.983586,49.177231,2.827414e-259,48.441724,49.912739,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ECRECOVER,reth,32.967906,3.510138e-02,0.992381,0.992338,47.624125,1.856072e-190,47.006893,48.241357,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ECRECOVER,nethermind,29.676244,8.261923e-01,0.266975,0.264430,27.858518,3.401549e-21,22.504716,33.212321,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ECADD,geth,21.422265,7.298816e-01,0.365409,0.364311,1.823291,4.559115e-59,1.626996,2.019585,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,BLS12_G2MSM,nethermind,1147.098934,6.386812e-150,0.786493,0.785904,240.842626,5.805885e-139,225.919945,255.765308,...,NaN,NaN,NaN,NaN,NaN,NaN,-7.865087,2.375608e-63,-8.695906,-7.034268
60,BLS12_PAIRING_CHECK,geth,1056.552526,1.367071e-82,0.320040,0.318411,729.692540,1.206529e-69,655.963017,803.422063,...,NaN,NaN,16.148625,0.000003,9.390380,22.906869,NaN,NaN,NaN,NaN
61,BLS12_PAIRING_CHECK,besu,1053.694424,1.045987e-82,0.332426,0.330827,745.257703,1.728676e-72,671.793159,818.722247,...,NaN,NaN,15.436334,0.000008,8.702378,22.170290,NaN,NaN,NaN,NaN
62,BLS12_PAIRING_CHECK,reth,1076.218236,2.712773e-20,0.228176,0.223484,823.791547,3.083161e-20,659.307526,988.275568,...,NaN,NaN,34.693521,0.000008,19.658322,49.728720,NaN,NaN,NaN,NaN


In [40]:
all_opcodes = set(operation_types.PRECOMPILES)
non_missing_operations = all_opcodes - missing_operations

estimated_operations_without_errors = set(results_df["opcode"].unique().tolist())
estimated_operations_with_errors = (
    non_missing_operations - estimated_operations_without_errors
)

print(
    f"Number of operations with estimation errors: {len(estimated_operations_with_errors)}"
)
estimated_operations_with_errors

Number of operations with estimation errors: 0


set()

In [41]:
estimation_by_client = results_df.groupby("opcode")["client"].nunique()
max_clients = estimation_by_client.max()
print(f"There is a max of {max_clients} clients with estimations.")
opcodes_with_missing_clients = estimation_by_client[estimation_by_client<5].index
print(
    f"Number of operations with estimation errors: {len(opcodes_with_missing_clients)}"
)
opcodes_with_missing_clients

There is a max of 4 clients with estimations.
Number of operations with estimation errors: 16


Index(['BLAKE2F', 'BLS12_G1ADD', 'BLS12_G1MSM', 'BLS12_G2ADD', 'BLS12_G2MSM',
       'BLS12_MAP_FP2_TO_G2', 'BLS12_MAP_FP_TO_G1', 'BLS12_PAIRING_CHECK',
       'ECADD', 'ECMUL', 'ECPAIRING', 'ECRECOVER', 'IDENTITY',
       'POINT_EVALUATION', 'RIPEMD-160', 'SHA2-256'],
      dtype='object', name='opcode')

#### Negative features

In [42]:
pvalue_cols = [i for i in results_df.columns if ("_pvalue" in i) & ("intercept" not in i)]
coef_cols = [i.replace("_pvalue", "") for i in pvalue_cols]
coef_cols

['slope', 'size', 'num_pairs', 'k']

In [43]:
features_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=coef_cols,
    var_name="feature",
    value_name="value"
).dropna()
features_df[features_df["value"]<0].sort_values("opcode")

,opcode,client,feature,value
48,BLAKE2F,geth,slope,-0.244322
50,BLAKE2F,reth,slope,-1.461593
53,BLS12_G1MSM,besu,slope,-3.581611
247,BLS12_G1MSM,nethermind,k,-3.396561
57,BLS12_G2MSM,besu,slope,-29.904029
248,BLS12_G2MSM,geth,k,-3.808468
249,BLS12_G2MSM,besu,k,-4.098974
250,BLS12_G2MSM,reth,k,-4.857140
251,BLS12_G2MSM,nethermind,k,-7.865087


#### Non-significant features (by p-value)

In [44]:
results_df[results_df["slope_pvalue"]>0.05]

pvalue_df = results_df.melt(
    id_vars=["opcode", "client"],
    value_vars=pvalue_cols,
    var_name="feature",
    value_name="pvalue"
).dropna()
pvalue_df[pvalue_df["pvalue"]>0.05].sort_values("opcode")

,opcode,client,feature,pvalue
53,BLS12_G1MSM,besu,slope_pvalue,0.491842
245,BLS12_G1MSM,besu,k_pvalue,0.081202
246,BLS12_G1MSM,reth,k_pvalue,0.522129


#### Small r-squared values (besides non-significant slopes)

In [45]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]

,opcode,client,intercept,intercept_pvalue,rsquared,rsquared_adj,slope,slope_pvalue,slope_conf_int_low,slope_conf_int_high,...,size_conf_int_low,size_conf_int_high,num_pairs,num_pairs_pvalue,num_pairs_conf_int_low,num_pairs_conf_int_high,k,k_pvalue,k_conf_int_low,k_conf_int_high
3,ECRECOVER,nethermind,29.676244,8.261923e-01,0.266975,0.264430,27.858518,3.401549e-21,22.504716,33.212321,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ECADD,geth,21.422265,7.298816e-01,0.365409,0.364311,1.823291,4.559115e-59,1.626996,2.019585,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ECADD,reth,36.303643,7.456111e-01,0.307045,0.305110,2.269611,2.315161e-30,1.915222,2.623999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ECADD,nethermind,25.599221,5.722332e-01,0.328562,0.327401,1.227864,5.829382e-52,1.084467,1.371261,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ECMUL,besu,28.422875,5.505835e-01,0.134689,0.133692,22.292402,3.957634e-29,18.528219,26.056586,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,ECMUL,reth,30.407408,7.641159e-01,0.114108,0.112462,33.969641,7.031478e-16,25.953640,41.985642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,ECMUL,nethermind,28.311760,4.059944e-01,0.075325,0.074260,11.537692,1.688030e-16,8.844686,14.230698,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,POINT_EVALUATION,nethermind,17.268679,9.759227e-01,0.259735,0.257165,926.161065,1.419678e-20,744.820232,1107.501898,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,BLS12_G1ADD,nethermind,16.424861,8.711066e-01,0.334893,0.332584,3.925773,2.528558e-27,3.284122,4.567423,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,BLS12_G2ADD,nethermind,8.222951,9.327325e-01,0.313087,0.310702,5.990919,2.720688e-25,4.961735,7.020102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
results_df[(results_df["rsquared"]<0.5) & (~results_df["opcode"].isin(non_significant_ops))]["opcode"].unique()

array(['ECRECOVER', 'ECADD', 'ECMUL', 'POINT_EVALUATION', 'BLS12_G1ADD',
       'BLS12_G2ADD', 'BLS12_MAP_FP_TO_G1', 'BLS12_MAP_FP2_TO_G2',
       'RIPEMD-160', 'ECPAIRING', 'BLAKE2F', 'BLS12_G1MSM', 'BLS12_G2MSM',
       'BLS12_PAIRING_CHECK'], dtype=object)